In [ ]:
import os

input_dir = "/home/jovyan/work/DataSets"



print("Files found:")
print(os.listdir(input_dir))


input_dir = "/home/jovyan/work/DataSets/MIX"



print("Files found:")
print(os.listdir(input_dir))

Files found:
['MIX_Seed1', 'MIX_Seed2', 'MIX_Seed3', 'OTX_Seed1', 'OTX_Seed2', 'OTX_Seed3', 'SYN_Seed1', 'SYN_Seed2', 'SYN_Seed3', 'Testing.jsonl']
Files found:
['Combined_Seed3_Test_15.jsonl', 'Combined_Seed3_Train_70.jsonl', 'Combined_Seed3_Valid_15.jsonl']


In [2]:
!pip install datasets

In [4]:
import datasets
from datasets import load_dataset

# Define the path to your uploaded files
data_files = {
    "train": os.path.join(input_dir, "Combined_Seed3_Train_70.jsonl"),
    "test": os.path.join(input_dir, "Combined_Seed3_Test_15.jsonl"),
    "validate": os.path.join(input_dir, "Combined_Seed3_Valid_15.jsonl")
}

# Load the dataset
ladder_dataset = load_dataset("json", data_files=data_files)

# Display the dataset structure
ladder_dataset

Generating train split: 10770 examples [00:00, 329850.19 examples/s]
Generating test split: 2309 examples [00:00, 245001.09 examples/s]
Generating validate split: 2308 examples [00:00, 236154.70 examples/s]


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 10770
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2309
    })
    validate: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2308
    })
})

In [5]:
!pip install -q datasets evaluate transformers[sentencepiece] seqeval
!apt install -y git-lfs

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [6]:
label_names= ladder_dataset["train"].features["ner_tags"].feature

label_names

ladder_dataset["train"][0]["ner_tags"]

unique_labels = set()
for example in ladder_dataset["train"]:
    unique_labels.update(example["ner_tags"])

label_names = sorted(unique_labels)
print(label_names)

['B-MalLibrary', 'B-MalProcess', 'B-TTPArgument', 'B-TargetLibrary', 'B-TargetProcess', 'I-MalLibrary', 'I-TargetProcess', 'O']


In [7]:
# Set Entity Classes
from datasets.features import ClassLabel, Sequence


entity_classes = [
    'B-MalProcess', 'B-TargetProcess', 'B-MalLibrary', 'B-TargetLibrary', 'B-TTPArgument',
    'I-MalProcess', 'I-TargetProcess', 'I-MalLibrary', 'I-TargetLibrary','I-TTPArgument',
    'O'
]
# is there a specific order how to order entity classes??

ner_feature = Sequence(ClassLabel(names=entity_classes))
ladder_dataset = ladder_dataset.cast_column("ner_tags", ner_feature)

# SET LABEL_NAMES
label_names = ner_feature.feature.names
label_names

id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

Casting the dataset: 100%|███████████████████████████████████████████████████████████████████████| 2308/2308 [00:00<00:00, 106965.16 examples/s]


In [8]:
id2label

{0: 'B-MalProcess',
 1: 'B-TargetProcess',
 2: 'B-MalLibrary',
 3: 'B-TargetLibrary',
 4: 'B-TTPArgument',
 5: 'I-MalProcess',
 6: 'I-TargetProcess',
 7: 'I-MalLibrary',
 8: 'I-TargetLibrary',
 9: 'I-TTPArgument',
 10: 'O'}

In [9]:
label2id

{'B-MalProcess': 0,
 'B-TargetProcess': 1,
 'B-MalLibrary': 2,
 'B-TargetLibrary': 3,
 'B-TTPArgument': 4,
 'I-MalProcess': 5,
 'I-TargetProcess': 6,
 'I-MalLibrary': 7,
 'I-TargetLibrary': 8,
 'I-TTPArgument': 9,
 'O': 10}

In [10]:
!pip install datasets evaluate transformers
!pip install accelerate
# To run the training on TPU, you will need to uncomment the following line:
# !pip install cloud-tpu-client==0.10 torch==1.9.0 https://storage.googleapis.com/tpu-pytorch/wheels/torch_xla-1.9-cp37-cp37m-linux_x86_64.whl
!apt install git-lfs
!pip install seqeval
!pip install datasets evaluate transformers[sentencepiece]

import evaluate
metric = evaluate.load("seqeval")

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [11]:
import numpy as np

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }
    

In [12]:
# 1. SET SECURE BERT AS CHECKPOINT AND SET AUTO TOKENIZER

from transformers import AutoTokenizer, AutoModelForTokenClassification

checkpoint = "xlm-roberta-large"

tokenizer = AutoTokenizer.from_pretrained(checkpoint, add_prefix_space=True)

In [13]:
# SET BASE MODEL AND USE AUTO TOKENIZER >> SECURE BERT

from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    checkpoint,
    id2label=id2label,
    label2id=label2id,
)

model.config.num_labels

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


11

In [14]:
# ============================================================================
# CORRECTED ALIGN FUNCTION - labels are now integers after casting
# ============================================================================
def align_labels_with_tokens(labels, word_ids):
    """
    FIXED VERSION: Properly converts B- to I- tags for subword tokens
    using string matching instead of modulo arithmetic
    """
    new_labels = []
    current_word = None
    
    for word_id in word_ids:
        if word_id != current_word:
            # Start of a new word!
            current_word = word_id
            label = -100 if word_id is None else labels[word_id]
            new_labels.append(label)
            
        elif word_id is None:
            # Special token
            new_labels.append(-100)
            
        else:
            # Same word as previous token (subword continuation)
            label = labels[word_id]
            
            # Convert B- tag to I- tag for subword tokens
            label_name = id2label[label]
            if label_name.startswith('B-'):
                entity_type = label_name[2:]  # Remove 'B-'
                i_label_name = f'I-{entity_type}'
                label = label2id[i_label_name]
            
            new_labels.append(label)
    
    return new_labels


In [15]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True, 
        padding=False, 
        max_length=128
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))
    
    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

print("\nTokenizing test set with CORRECTED align function...")
tokenized_test = ladder_dataset["test"].map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=ladder_dataset["test"].column_names
)



Tokenizing test set with CORRECTED align function...


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████| 2309/2309 [00:00<00:00, 19418.31 examples/s]


In [16]:
tokenized_datasets = ladder_dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=ladder_dataset["train"].column_names
)

Map: 100%|████████████████████████████████████████████████████████████████████████████████████████| 2308/2308 [00:00<00:00, 27518.02 examples/s]


In [17]:
example = tokenized_datasets["train"][0]

print (tokenizer.convert_ids_to_tokens(example["input_ids"]))
print (example["labels"])


['<s>', '▁Critic', 'al', '▁system', '▁libra', 'ry', '▁ad', 'va', 'pi', '32', '.', 'd', 'll', '▁was', '▁target', 'ed', '▁for', '▁function', '▁hook', 'ing', '▁', '.', '</s>']
[-100, 10, 10, 10, 10, 10, 3, 8, 8, 8, 8, 8, 8, 10, 10, 10, 10, 10, 10, 10, 10, 10, -100]


In [18]:
# Inspect one tokenized sample
example = tokenized_datasets["train"][0]

# Decode tokens back to words
print("TOKENS:")
print(tokenizer.convert_ids_to_tokens(example["input_ids"]))

print("\nLABEL IDS:")
print(example["labels"])

# Optional: human-readable labels
print("\nLABEL NAMES:")
print([id2label[l] if l != -100 else "IGNORED" for l in example["labels"]])

TOKENS:
['<s>', '▁Critic', 'al', '▁system', '▁libra', 'ry', '▁ad', 'va', 'pi', '32', '.', 'd', 'll', '▁was', '▁target', 'ed', '▁for', '▁function', '▁hook', 'ing', '▁', '.', '</s>']

LABEL IDS:
[-100, 10, 10, 10, 10, 10, 3, 8, 8, 8, 8, 8, 8, 10, 10, 10, 10, 10, 10, 10, 10, 10, -100]

LABEL NAMES:
['IGNORED', 'O', 'O', 'O', 'O', 'O', 'B-TargetLibrary', 'I-TargetLibrary', 'I-TargetLibrary', 'I-TargetLibrary', 'I-TargetLibrary', 'I-TargetLibrary', 'I-TargetLibrary', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'IGNORED']


In [21]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [22]:
# SET TRAINING ARGUMENTS
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir = "/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed3/",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    greater_is_better = True, #what is this?
    learning_rate=1e-6,
    num_train_epochs=20,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    weight_decay=0.0,
    push_to_hub=False,
    #per_device_train_batch_size=4,
    #per_device_eval_batch_size=4,
    report_to = [] #removes wandb
)

In [23]:
#SET TRAINER

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validate"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

/tmp/ipykernel_2049/2322743198.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [24]:
# TRAIN
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.583900,0.061526,0.805985,0.863495,0.833749,0.987217
2,0.066900,0.026945,0.887374,0.908480,0.897803,0.993023
3,0.028000,0.019636,0.892370,0.913133,0.902632,0.994894
4,0.020900,0.014734,0.941355,0.962771,0.951943,0.996293
5,0.016400,0.010919,0.958081,0.980869,0.969341,0.997342
6,0.010800,0.009092,0.966463,0.983454,0.974885,0.997814
7,0.009300,0.010435,0.969959,0.985005,0.977424,0.997744
8,0.006300,0.012468,0.971501,0.987073,0.979225,0.996957
9,0.004700,0.009906,0.979540,0.990176,0.984829,0.997692
10,0.006500,0.008898,0.980533,0.989659,0.985075,0.998076


TrainOutput(global_step=13480, training_loss=0.031135367759080598, metrics={'train_runtime': 2001.9037, 'train_samples_per_second': 107.598, 'train_steps_per_second': 6.734, 'total_flos': 2.7125355149753616e+16, 'train_loss': 0.031135367759080598, 'epoch': 20.0})

In [ ]:
# =================================
# Entity-Level Evaluation (seqeval)
# =================================
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, DataCollatorForTokenClassification
from seqeval.metrics import classification_report as seqeval_report, f1_score, precision_score, recall_score
import numpy as np

# ---------------------------------------------------------------
# 1) Specify the checkpoint  to evaluate
# ---------------------------------------------------------------

model_checkpoint = "/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed3/checkpoint-13480"


# ---------------------------------------------------------------
# 2) Load model and tokenizer from checkpoint
# ---------------------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)

# ---------------------------------------------------------------
# 3) Prepare data
# ---------------------------------------------------------------
# ladder_dataset → your DatasetDict with train/validate/test splits
# tokenized_datasets → the tokenized version built with your tokenize_and_align_labels()

data_collator = DataCollatorForTokenClassification(tokenizer)

# label mapping (must match model config)
label_names = list(model.config.id2label.values())

# ---------------------------------------------------------------
# 4) Build eval-only Trainer and predict on test split
# ---------------------------------------------------------------
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="tmp-eval",
    per_device_eval_batch_size=32,
    report_to=[],
)

eval_trainer = Trainer(
    model=model,
    args=args,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

pred_out = eval_trainer.predict(tokenized_datasets["test"])
logits = pred_out.predictions
label_ids = pred_out.label_ids
pred_ids = np.argmax(logits, axis=-1)

# ---------------------------------------------------------------
# 5) Convert label IDs → tag strings, drop masked (-100) tokens
# ---------------------------------------------------------------
true_tags, pred_tags = [], []
for p_seq, l_seq in zip(pred_ids, label_ids):
    t_seq, p_seq_str = [], []
    for p, l in zip(p_seq, l_seq):
        if l == -100:
            continue
        t_seq.append(label_names[l])
        p_seq_str.append(label_names[p])
    true_tags.append(t_seq)
    pred_tags.append(p_seq_str)

# ---------------------------------------------------------------
# 6) LADDER-style entity-level metrics
# ---------------------------------------------------------------
print("\n=== Entity-level (seqeval) metrics —  ===")
print (model_checkpoint)
print(f"Precision: {precision_score(true_tags, pred_tags):.4f}")
print(f"Recall:    {recall_score(true_tags, pred_tags):.4f}")
print(f"F1 Score:  {f1_score(true_tags, pred_tags):.4f}")

print("\n--- Per-class breakdown (entity-level) ---")
print(seqeval_report(true_tags, pred_tags, digits=4))


The tokenizer you are loading from '/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed3/checkpoint-13480' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
/tmp/ipykernel_2049/3691754772.py:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  eval_trainer = Trainer(



=== Entity-level (seqeval) metrics —  ===
/home/jovyan/work/Results/XLM-Roberta-large/MIXSeed3/checkpoint-13480
Precision: 0.9689
Recall:    0.9857
F1 Score:  0.9772

--- Per-class breakdown (entity-level) ---
               precision    recall  f1-score   support

   MalLibrary     0.9544    0.9611    0.9577       283
   MalProcess     0.9359    0.9903    0.9624       413
  TTPArgument     0.9946    0.9928    0.9937       556
TargetLibrary     0.9816    0.9907    0.9861       215
TargetProcess     0.9721    0.9858    0.9789       494

    micro avg     0.9689    0.9857    0.9772      1961
    macro avg     0.9677    0.9842    0.9758      1961
 weighted avg     0.9693    0.9857    0.9773      1961

